In [10]:
# 加载环境变量
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import json
import os
from rich import print as rprint

load_dotenv()

True

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [11]:
# 导入langchain的初始化模型的函数

model = init_chat_model(
    model="deepseek-v4-pro",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0.5,
    max_tokens=1024,
    max_retries=3,
)
rprint(type(model))
rprint(json.dumps(model.profile, indent=4))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>

{
    "name": "DeepSeek V4 Pro",
    "release_date": "2026-04-24",
    "last_updated": "2026-04-24",
    "open_weights": true,
    "max_input_tokens": 1000000,
    "max_output_tokens": 384000,
    "text_inputs": true,
    "image_inputs": false,
    "audio_inputs": false,
    "video_inputs": false,
    "text_outputs": true,
    "image_outputs": false,
    "audio_outputs": false,
    "video_outputs": false,
    "reasoning_output": true,
    "tool_calling": true,
    "structured_output": true,
    "attachment": false,
    "temperature": true
}

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="deepseek-v4-pro",
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [ ]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")
print(response)

In [ ]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [ ]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [ ]:
# 打印stream类型
print(type(stream))

In [ ]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [ ]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [ ]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-v4-pro")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

### 流式访问


In [ ]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type(messages))

In [ ]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if
token.content:  # Check if there's actual content
print(token.content, end="", flush=True)  # Print token